# Week 8, Day 4: Partial derivatives and gradients: many inputs at once

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w008/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A numerical gradient for anything

Write `numerical_gradient(f, w, h=1e-5)` that returns the vector of central differences, nudging one entry of `w` at a time. Then write `grad_L(w)`, the analytic gradient of the two-weight squared error above (the average error times each input), and check it against the numerical one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(8)
X = rng.normal(size=(100, 2))
y = X @ np.array([1.2, -0.7]) + 0.3 * rng.normal(size=100)

def L(w):
    return 0.5 * np.mean((X @ w - y) ** 2)

def numerical_gradient(f, w, h=1e-5):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = ...
    return g

def grad_L(w):
    return ...

w = np.array([0.5, 0.5])
print("analytic", grad_L(w).round(5), " numerical", numerical_gradient(L, w).round(5))

In [ ]:
import numpy as np
def test_quadratic():
    "numerical_gradient of w1^2 + 3 w1 w2 matches (2 w1 + 3 w2, 3 w1)"
    f = lambda w: w[0] ** 2 + 3 * w[0] * w[1]
    assert np.allclose(numerical_gradient(f, np.array([1.0, 2.0])), [8.0, 3.0], atol=1e-6)

def test_loss():
    "grad_L matches the numerical gradient of the loss at several points"
    for w in [np.array([0.0, 0.0]), np.array([1.5, -0.5]), np.array([-2.0, 3.0])]:
        assert np.allclose(grad_L(w), numerical_gradient(L, w), atol=1e-6)

test_quadratic()
test_loss()
print("All checks passed")

### Exercise: Probe a model neuron

For a 100-pixel model neuron, write `grad_x(x)`, the gradient of its rate with respect to the stimulus. Starting from the image `x0`, compare the rise in rate produced by a change of length 0.5 along the gradient (`gain_best`) with 200 random changes of the same length (`gains_random`).

In [ ]:
n_pix = 100
w_rf = np.sin(np.linspace(0, 3 * np.pi, n_pix)) * np.exp(-np.linspace(-2, 2, n_pix) ** 2)   # a receptive field
def sigmoid(u): return 1 / (1 + np.exp(-u))
def rate(x): return 50 * sigmoid(w_rf @ x - 1)

def grad_x(x):
    u = w_rf @ x - 1
    return ...

x0 = rng.normal(size=n_pix) * 0.1
g = grad_x(x0)
step = 0.5 * g / np.linalg.norm(g)
gain_best = rate(x0 + step) - rate(x0)
gains_random = []
for _ in range(200):
    d = rng.normal(size=n_pix); d = 0.5 * d / np.linalg.norm(d)
    gains_random.append(rate(x0 + d) - rate(x0))

plt.figure(figsize=(5, 3)); plt.hist(gains_random, bins=25, label="random changes"); plt.axvline(gain_best, color="C1", label="along the gradient")
plt.xlabel("change in rate (Hz)"); plt.legend(frameon=False); plt.show()
print("best", round(gain_best, 2), "Hz;  best random", round(max(gains_random), 2), "Hz")

In [ ]:
import numpy as np
def test_grad():
    "grad_x matches the numerical gradient of the neuron's rate"
    assert np.allclose(grad_x(x0), numerical_gradient(rate, x0), atol=1e-6)

def test_best():
    "the change along the gradient raises the rate more than every random change of the same size"
    assert gain_best > max(gains_random)

def test_parallel():
    "the gradient is parallel to the receptive field w"
    g = grad_x(x0)
    assert np.isclose(abs(g @ w_rf) / (np.linalg.norm(g) * np.linalg.norm(w_rf)), 1)

test_grad()
test_best()
test_parallel()
print("All checks passed")

### Exercise: Discrimination by a population

Write `pop_fisher(theta, prefs, T)` for a population of Gaussian-tuned Poisson neurons with preferred orientations `prefs` (use `tune` and `dtune` below, which wrap orientation round 180°). For populations of N = 10, 40, 160 and 640 neurons with evenly spaced preferences, compute the discrimination threshold $1/\sqrt{I}$ at 60° in a 0.5 s window, and plot it against N on log-log axes.

In [ ]:
def wrap(d):
    return (d + 90) % 180 - 90          # orientation difference in (-90, 90]

def tune(theta, pref, b=5.0, A=40.0, s=25.0):
    return b + A * np.exp(-wrap(theta - pref) ** 2 / (2 * s ** 2))

def dtune(theta, pref, b=5.0, A=40.0, s=25.0):
    d = wrap(theta - pref)
    return -A * d / s ** 2 * np.exp(-d ** 2 / (2 * s ** 2))

def pop_fisher(theta, prefs, T):
    return ...

Ns = [10, 40, 160, 640]
thresholds = [1 / np.sqrt(pop_fisher(60.0, np.linspace(0, 180, N, endpoint=False), 0.5)) for N in Ns]
plt.figure(figsize=(5, 3)); plt.loglog(Ns, thresholds, "o-"); plt.xlabel("neurons N"); plt.ylabel("threshold (degrees)"); plt.show()
print(np.round(thresholds, 3))

In [ ]:
import numpy as np
def test_info():
    "pop_fisher sums each neuron's T f'^2 / f"
    prefs = np.array([30.0, 90.0])
    expected = sum(0.5 * dtune(60.0, p) ** 2 / tune(60.0, p) for p in prefs)
    assert np.isclose(pop_fisher(60.0, prefs, 0.5), expected)

def test_scaling():
    "the threshold falls roughly like 1 / sqrt(N)"
    ratio = thresholds[0] / thresholds[-1]
    assert abs(ratio - np.sqrt(Ns[-1] / Ns[0])) / np.sqrt(Ns[-1] / Ns[0]) < 0.15

test_info()
test_scaling()
print("All checks passed")